In [ ]:
import pandas as pd
df = pd.read_csv("/content/merged_dataset.csv")
print(df.head())

                                                Text  Label
0  ମମତା ବାନାର୍ଜୀ ଙ୍କ ଚୁନାଵ ପ୍ରଚାର ଓ ବିଲ୍ ପାସ୍ କୁ ...      1
1  ଧର୍ମେନ୍ଦ୍ର ପ୍ରଧାନ ଙ୍କ ବିଲ୍ ପାସ୍ ଓ ଚୁନାଵ ପ୍ରଚାର...      1
2  ଚୋରି ଚୋରି ଚୁପକେ ଚୁପକେ … ଭୋଟ୍ ଚୋରିକୁ ନେଇ ସରକାରଙ...      0
3  କୋଲୋଷ୍ଟ୍ରମ୍ ସପ୍ଲିମେଣ୍ଟ ଖାଇଲେ ବୟସ୍କ ଲୋକଙ୍କର ଗଟ୍...      1
4  ab de villiers ଇଂଲଣ୍ଡ ବିରୋଧରେ odi ସିରିଜ୍ରେ ଦଳ ...      1


In [ ]:
df.isnull().sum()

,0
Text,0
Label,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1986 entries, 0 to 1985
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Text    1986 non-null   object
 1   Label   1986 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 31.2+ KB


In [ ]:
df.duplicated().sum()

np.int64(103)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
import pandas as pd
import re
import os

In [ ]:
input_file = "/content/merged_dataset.csv"

df = pd.read_csv(input_file,encoding = "utf-8-sig",encoding_errors="ignore")

print("Before cleaning:",df.shape)
print(df["Label"].value_counts())


# keep only needed columns
df = df[["Text","Label"]]

df.dropna(inplace=True)
df["Text"] = df["Text"].astype(str)
df["Label"] = df["Label"].astype(int)

# ===============================
# 2. Remove Horoscope / Rashi news
# ===============================

horoscope_patterns = [
    "ରାଶି",
    "ରାଶିଫଳ",
    "ମେଷ ରାଶି",
    "ବୃଷ ରାଶି",
    "ମିଥୁନ ରାଶି",
    "କର୍କଟ ରାଶି",
    "ସିଂହ ରାଶି",
    "କନ୍ୟା ରାଶି",
    "ତୁଳା ରାଶି",
    "ବିଛା ରାଶି",
    "ଧନୁ ରାଶି",
    "ମକର ରାଶି",
    "କୁମ୍ଭ ରାଶି",
    "ମୀନ ରାଶି"
]

for pattern in horoscope_patterns:
    df = df[~df["Text"].str.contains(pattern, na=False)]

# ===============================
# 3. Remove clickbait / template fake patterns
# ===============================

bad_patterns = [
    "ବଡ ଖୁଲାସା",
    "ବଡ଼ ଖୁଲାସା",
    "ବଡ ଘଟଣା",
    "ବଡ଼ ଘଟଣା",
    "ବଡ ରହସ୍ୟ",
    "ନୂତନ ଖୁଲାସା",
    "ଭୟଙ୍କର",
    "ଚମତ୍କାର",
    "ଗୁପ୍ତ ସୂଚନା",
    "ହଠାତ୍",
    "ଆଶ୍ଚର୍ୟଜନକ",
    "100% ସତ୍ୟ",
    "ଲିଙ୍କ କ୍ଲିକ୍",
    "ତୁରନ୍ତ ଶେୟର୍",
    "ଏହା ସତ୍ୟ କି ନୁହେଁ"
]

for pattern in bad_patterns:
    df = df[~df["Text"].str.contains(pattern, na=False)]

# ===============================
# 4. Remove very short text
# ===============================

df["length"] = df["Text"].str.len()

# only remove very small / useless text
df = df[df["length"] > 30]

df.drop(columns=["length"], inplace=True)

# ===============================
# 5. Remove duplicates
# ===============================

df.drop_duplicates(subset=["Text"], inplace=True)

print("\nAfter cleaning:", df.shape)
print(df["Label"].value_counts())

# Save cleaned dataset
df.to_csv("clean_step1_dataset.csv", index=False, encoding="utf-8-sig")
df.to_excel("clean_step1_dataset.xlsx", index=False)

print("\n✅ Saved clean_step1_dataset.csv and clean_step1_dataset.xlsx")

# ===============================
# 6. Balance dataset
# ===============================

real_df = df[df["Label"] == 0]
fake_df = df[df["Label"] == 1]

min_count = min(len(real_df), len(fake_df))

real_df = real_df.sample(n=min_count, random_state=42)
fake_df = fake_df.sample(n=min_count, random_state=42)

df_balanced = pd.concat([real_df, fake_df])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter balancing:")
print(df_balanced.shape)
print(df_balanced["Label"].value_counts())

# ===============================
# 7. Save final balanced dataset
# ===============================

df_balanced.to_csv("last_balanced_dataset.csv", index=False, encoding="utf-8-sig")
df_balanced.to_excel("last_balanced_dataset.xlsx", index=False)

print("\n✅ Saved last_balanced_dataset.csv and last_balanced_dataset.xlsx")

Before cleaning: (1986, 2)
Label
1    998
0    988
Name: count, dtype: int64

After cleaning: (1794, 2)
Label
1    932
0    862
Name: count, dtype: int64

✅ Saved clean_step1_dataset.csv and clean_step1_dataset.xlsx

After balancing:
(1724, 2)
Label
0    862
1    862
Name: count, dtype: int64

✅ Saved last_balanced_dataset.csv and last_balanced_dataset.xlsx


In [ ]:
import re
import pandas as pd

# ===============================
# 1. Load dataset safely
# ===============================

input_file = "last_balanced_dataset.csv"

df = pd.read_csv(
    input_file,
    encoding="utf-8-sig",
    encoding_errors="ignore"
)

df = df[["Text", "Label"]]
df.dropna(inplace=True)

df["Text"] = df["Text"].astype(str)
df["Label"] = df["Label"].astype(int)

print("Original:")
print(df.shape)
print(df["Label"].value_counts())

# ===============================
# 2. Basic text cleaning
# ===============================

def clean_basic(text):
    text = str(text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # remove repeated punctuation
    text = re.sub(r"[.]{2,}", ".", text)
    text = re.sub(r"[-]{2,}", "-", text)

    return text.strip()

df["Text"] = df["Text"].apply(clean_basic)

# ===============================
# 3. Remove footer / website junk
# ===============================

footer_patterns = [
    "join and get latest",
    "enter your email",
    "kanak news is odisha",
    "all rights reserved",
    "quick links",
    "developed by",
    "subscribe to our newsletter",
    "dharitri",
    "read latest news",
]

def remove_footer(text):
    text_lower = text.lower()
    for p in footer_patterns:
        idx = text_lower.find(p)
        if idx != -1:
            text = text[:idx]
            break
    return text.strip()

df["Text"] = df["Text"].apply(remove_footer)

# ===============================
# 4. Remove horoscope rows
# ===============================

horoscope_patterns = [
    "ରାଶିଫଳ",
    "ମେଷ ରାଶି",
    "ବୃଷ ରାଶି",
    "ମିଥୁନ ରାଶି",
    "କର୍କଟ ରାଶି",
    "ସିଂହ ରାଶି",
    "କନ୍ୟା ରାଶି",
    "ତୁଳା ରାଶି",
    "ବିଛା ରାଶି",
    "ଧନୁ ରାଶି",
    "ମକର ରାଶି",
    "କୁମ୍ଭ ରାଶି",
    "ମୀନ ରାଶି",
]

for p in horoscope_patterns:
    df = df[~df["Text"].str.contains(p, na=False)]

# ===============================
# 5. Remove bad fake templates
# ===============================

bad_fake_patterns = [
    "ବଡ ଖୁଲାସା",
    "ବଡ଼ ଖୁଲାସା",
    "ବଡ ଘଟଣା",
    "ବଡ଼ ଘଟଣା",
    "ବଡ ପରିବର୍ତ୍ତନ",
    "ବଡ଼ ପରିବର୍ତ୍ତନ",
    "ଅପୂର୍ବ ଖବର",
    "ଭୟଙ୍କର",
    "ଚମତ୍କାର",
    "ଗୁପ୍ତ ସୂଚନା",
    "ହଠାତ୍",
    "ଆଶ୍ଚର୍ୟଜନକ",
    "ଅପ୍ରମାଣିତ ସୂଚନା",
    "100% ସତ୍ୟ",
    "ତୁରନ୍ତ ଶେୟର",
    "ଲିଙ୍କ କ୍ଲିକ୍",
    "ଏହା ସତ୍ୟ କି ନୁହେଁ",
    "ଫ୍ୟାନ୍‌ମାନେ ଆଶ୍ଚର୍ଯ୍ୟ",
    "ଲୋକମାନେ ବହୁତ ଶେୟର",
]

for p in bad_fake_patterns:
    df = df[~((df["Label"] == 1) & (df["Text"].str.contains(p, na=False)))]

# ===============================
# 6. Remove repeated entertainment fake templates
# ===============================

entertainment_patterns = [
    "ଦର୍ଶକମାନଙ୍କ ଆଗ୍ରହ ବଢିଛି",
    "ମିଶ୍ର ପ୍ରତିକ୍ରିୟା",
    "ସୋସିଆଲ ମିଡିଆରେ ଚର୍ଚ୍ଚା",
    "ବେଶ ଚର୍ଚ୍ଚା ଚାଲିଛି",
    "ନୂତନ ଗୀତ",
    "ନୂତନ ସିନେମା",
    "ନୂତନ ୱେବ ସିରିଜ୍",
    "ଓଟିଟି ରିଲିଜ୍",
    "ପ୍ରମୋସନ୍",
]

for p in entertainment_patterns:
    df = df[~((df["Label"] == 1) & (df["Text"].str.contains(p, na=False)))]

# ===============================
# 7. Remove random crime template fake
# ===============================

crime_template_patterns = [
    "ଙ୍କ ନାମରେ ମାମଲା",
    "ସ୍ଥାନୀୟ ଅଞ୍ଚଳରେ ଏହା ନେଇ ଭୟ",
    "ପୋଲିସ .* ସନ୍ଦର୍ଭରେ ତଦନ୍ତ",
]

for p in crime_template_patterns:
    df = df[~((df["Label"] == 1) & (df["Text"].str.contains(p, regex=True, na=False)))]

# ===============================
# 8. Remove sports random fake templates
# ===============================

sports_fake_patterns = [
    "ମ୍ୟାନ୍ ଅଫ୍ ଦି ମ୍ୟାଚ୍",
    "ମ୍ୟାନ୍ ଅଫ୍ ଦି ସିରିଜ୍",
    "ରେକର୍ଡ ସୃଷ୍ଟି କଲେ",
]

for p in sports_fake_patterns:
    df = df[~((df["Label"] == 1) & (df["Text"].str.contains(p, na=False)))]

# ===============================
# 9. Remove very short rows
# ===============================

df["char_len"] = df["Text"].str.len()
df = df[df["char_len"] >= 40]

# ===============================
# 10. Trim very long rows
# ===============================

def limit_words(text, max_words=180):
    words = str(text).split()
    return " ".join(words[:max_words])

df["Text"] = df["Text"].apply(lambda x: limit_words(x, 180))

# Remove duplicates again after trimming
df.drop_duplicates(subset=["Text"], inplace=True)

# ===============================
# 11. Remove rows with too much English
# ===============================

def english_ratio(text):
    text = str(text)
    eng_chars = len(re.findall(r"[A-Za-z]", text))
    total_chars = max(len(text), 1)
    return eng_chars / total_chars

df["eng_ratio"] = df["Text"].apply(english_ratio)

# Keep mixed text only if English is not too dominant
df = df[df["eng_ratio"] <= 0.35]

df.drop(columns=["char_len", "eng_ratio"], inplace=True)

# ===============================
# 12. Balance dataset
# ===============================

real_df = df[df["Label"] == 0]
fake_df = df[df["Label"] == 1]

min_count = min(len(real_df), len(fake_df))

real_df = real_df.sample(n=min_count, random_state=42)
fake_df = fake_df.sample(n=min_count, random_state=42)

df_final = pd.concat([real_df, fake_df])
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter advanced cleaning and balancing:")
print(df_final.shape)
print(df_final["Label"].value_counts())

# ===============================
# 13. Save final files
# ===============================

df_final.to_csv("advanced_clean_balanced_dataset.csv", index=False, encoding="utf-8-sig")
df_final.to_excel("advanced_clean_balanced_dataset.xlsx", index=False)

print("\n✅ Saved:")
print("advanced_clean_balanced_dataset.csv")
print("advanced_clean_balanced_dataset.xlsx")

Original:
(1724, 2)
Label
0    862
1    862
Name: count, dtype: int64

After advanced cleaning and balancing:
(1692, 2)
Label
1    846
0    846
Name: count, dtype: int64

✅ Saved:
advanced_clean_balanced_dataset.csv
advanced_clean_balanced_dataset.xlsx


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("advanced_clean_balanced_dataset.csv", encoding="utf-8-sig")

X = df["Text"]
y = df["Label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

pd.DataFrame({"Text": X_train, "Label": y_train}).to_csv("C:\Users\HP\Desktop\Fake_news_detector\backend\train_expanded.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"Text": X_val, "Label": y_val}).to_csv("val.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"Text": X_test, "Label": y_test}).to_csv("test.csv", index=False, encoding="utf-8-sig")

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 1184
Validation: 254
Test: 254


In [ ]:
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =========================
# Load split datasets
# =========================
train_df = pd.read_csv("C:\Users\HP\Desktop\Fake_news_detector\backend\train_expanded.csv", encoding="utf-8-sig")
val_df = pd.read_csv("val.csv", encoding="utf-8-sig")
test_df = pd.read_csv("test.csv", encoding="utf-8-sig")

X_train = train_df["Text"].astype(str)
y_train = train_df["Label"]

X_val = val_df["Text"].astype(str)
y_val = val_df["Label"]

X_test = test_df["Text"].astype(str)
y_test = test_df["Label"]

# =========================
# TF-IDF Vectorization
# =========================
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

# =========================
# Models
# =========================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": LinearSVC()
}

# =========================
# Train and Evaluate
# =========================
for name, model in models.items():
    print("\n==============================")
    print(name)
    print("==============================")

    model.fit(X_train_tfidf, y_train)

    # Validation result
    val_pred = model.predict(X_val_tfidf)

    print("\nValidation Results:")
    print("Accuracy :", accuracy_score(y_val, val_pred))
    print("Precision:", precision_score(y_val, val_pred))
    print("Recall   :", recall_score(y_val, val_pred))
    print("F1-score :", f1_score(y_val, val_pred))
    print("\nClassification Report:")
    print(classification_report(y_val, val_pred, target_names=["Real", "Fake"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_val, val_pred))

    # Test result
    test_pred = model.predict(X_test_tfidf)

    print("\nTest Results:")
    print("Accuracy :", accuracy_score(y_test, test_pred))
    print("Precision:", precision_score(y_test, test_pred))
    print("Recall   :", recall_score(y_test, test_pred))
    print("F1-score :", f1_score(y_test, test_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred, target_names=["Real", "Fake"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, test_pred))

    # =========================
# Save TF-IDF and Models
# =========================

os.makedirs("saved_models", exist_ok=True)

# Save TF-IDF
with open("saved_models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("✅ TF-IDF saved")

# Save each model
for name, model in models.items():
    filename = name.lower().replace(" ", "_") + ".pkl"
    path = "saved_models/" + filename

    with open(path, "wb") as f:
        pickle.dump(model, f)

    print(f"✅ {name} saved as {filename}")


Logistic Regression

Validation Results:
Accuracy : 0.9251968503937008
Precision: 0.990909090909091
Recall   : 0.8582677165354331
F1-score : 0.919831223628692

Classification Report:
              precision    recall  f1-score   support

        Real       0.88      0.99      0.93       127
        Fake       0.99      0.86      0.92       127

    accuracy                           0.93       254
   macro avg       0.93      0.93      0.92       254
weighted avg       0.93      0.93      0.92       254

Confusion Matrix:
[[126   1]
 [ 18 109]]

Test Results:
Accuracy : 0.9330708661417323
Precision: 1.0
Recall   : 0.8661417322834646
F1-score : 0.9282700421940928

Classification Report:
              precision    recall  f1-score   support

        Real       0.88      1.00      0.94       127
        Fake       1.00      0.87      0.93       127

    accuracy                           0.93       254
   macro avg       0.94      0.93      0.93       254
weighted avg       0.94      0.9

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train)
X_val = vectorizer.transform(X_val)
X_test = vectorizer.transform(X_test)

In [ ]:
import pickle
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

print("Model trained successfully!")

# Prediction
y_pred = model.predict(X_test)

# Accuracy
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test,y_pred)

print("Accuracy:",accuracy)

Model trained successfully!
Accuracy: 0.9409448818897638


In [ ]:
# classification report
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94       127
           1       1.00      0.88      0.94       127

    accuracy                           0.94       254
   macro avg       0.95      0.94      0.94       254
weighted avg       0.95      0.94      0.94       254



In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[127   0]
 [ 15 112]]


In [ ]:
# save model
import pickle

pickle.dump(model,open("fake_news_model.pkl","wb"))
pickle.dump(vectorizer,open("tfidf_vectorizer.pkl","wb"))

print("Model saved successfully!")

Model saved successfully!


In [ ]:
# test with new news text
sample_news = ["ଏହା ଏକ ନୂତନ ଖବର "]

sample_vector = vectorizer.transform(sample_news)

prediction = model.predict(sample_vector)

print(prediction)

[1]


In [ ]:
import pickle

# Load the saved vectorizer and model
vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))
model = pickle.load(open("fake_news_model.pkl", "rb"))

news = input("Enter a news text: ")

news_vector = vectorizer.transform([news])

prediction = model.predict(news_vector)

print("Prediction: ", prediction[0])